In [87]:
import pandas as pd
import numpy as np
import math
from scipy.stats import ttest_ind, mannwhitneyu, shapiro
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import kstest, zscore


In [88]:
gen = pd.read_csv('/account001/mansi.chandra/clin_path/results_vae_corr_mod/predictions_decoded/test/samples/generated_samples_s5_929201_ControlGenerator_test.csv')
real = pd.read_csv('/account001/mansi.chandra/clin_path/repeat_test.csv')
threshold = pd.read_csv('/account001/mansi.chandra/clin_path/best_per_feature.csv')

In [89]:
#optimizing the prediction values to be consistent with real 

gen["TBIL(mg/dL)"] = pd.to_numeric(gen["TBIL(mg/dL)"], errors="coerce").round(2)
gen["RALB(g/dL)"] = pd.to_numeric(gen["RALB(g/dL)"], errors="coerce").round(2)
gen["AST(IU/L)"] = pd.to_numeric(gen["AST(IU/L)"], errors="coerce").round().astype(int)
gen["TP(g/dL)"] = pd.to_numeric(gen["TP(g/dL)"], errors="coerce").round(1)
gen["CRE(mg/dL)"] = pd.to_numeric(gen["CRE(mg/dL)"], errors="coerce").round(1)
gen["DBIL(mg/dL)"] = pd.to_numeric(gen["DBIL(mg/dL)"], errors="coerce").round(2)
gen["BUN(mg/dL)"] = pd.to_numeric(gen["BUN(mg/dL)"], errors="coerce").round().astype(int)
gen["K(meq/L)"] = pd.to_numeric(gen["K(meq/L)"], errors="coerce").round(2)
gen["GTP(IU/L)"] = pd.to_numeric(gen["GTP(IU/L)"], errors="coerce").round().astype(int)
gen["Ca(mg/dL)"] = pd.to_numeric(gen["Ca(mg/dL)"], errors="coerce").round(1)
gen["Cl(meq/L)"] = pd.to_numeric(gen["Cl(meq/L)"], errors="coerce").round(1)
gen["Na(meq/L)"] = pd.to_numeric(gen["Na(meq/L)"], errors="coerce").round(1)
gen["IP(mg/dL)"] = pd.to_numeric(gen["IP(mg/dL)"], errors="coerce").round(1)
gen["ALP(IU/L)"] = pd.to_numeric(gen["ALP(IU/L)"], errors="coerce").round().astype(int)
gen["ALT(IU/L)"] = pd.to_numeric(gen["ALT(IU/L)"], errors="coerce").round().astype(int)
gen["LDH(IU/L)"] = pd.to_numeric(gen["LDH(IU/L)"], errors="coerce").round().astype(int)
gen["RALB(g/dL)"] = pd.to_numeric(gen["RALB(g/dL)"], errors="coerce").round(1)


In [90]:
# 1) Which features to score? (columns 11 onward)
feature_cols = real.columns[11:]

results = []

# 2) Loop per (compound, time)
for (compound, time), grp in real.groupby(['COMPOUND_NAME','SACRIFICE_PERIOD']):
    # 2a) pull the control rows
    ctrl = grp[grp['DOSE_LEVEL']=='Control']
    if ctrl.shape[0] < 2:
        continue

    # 2b) precompute control means & SDs per feature
    ctrl_means = ctrl[feature_cols].mean()
    ctrl_sds   = ctrl[feature_cols].std(ddof=1)

    # 3) for each treatment dose
    for dose in ['High']:
        trt = grp[grp['DOSE_LEVEL']==dose]
        if trt.empty:
            continue

        # 4) compute z‑scores for each individual treatment sample
        for _, row in trt.iterrows():
            rec = {
                'compound':      compound,
                'time':          time,
                'dose':          dose,
                'INDIVIDUAL_ID': row['INDIVIDUAL_ID']
            }
            for f in feature_cols:
                μ = ctrl_means[f]
                σ = ctrl_sds[f]
                x = row[f]

                # cannot compute if SD is zero or missing
                if pd.isna(σ) or σ == 0:
                    rec[f'{f}_z'] = np.nan
                else:
                    rec[f'{f}_z'] = (x - μ) / σ

            results.append(rec)

# 5) Build the final DataFrame
results_df = pd.DataFrame(results)

In [91]:
# 1) Identify z-score columns
z_cols = [col for col in results_df.columns if col.endswith('_z')]
keys   = ['compound','time','dose']

# 2) Container
feature_counts = []

# 3) Per-feature (NA-aware) counting
for z in z_cols:
    s = results_df[z]

    # cell-level abnormal (strict > 2), keep NaN as NA (nullable boolean)
    abn = s.abs().gt(2)
    abn = abn.where(s.notna(), pd.NA).astype('boolean')

    # group to treatment-level: any True among non-NA; also track usable count
    grp = (
        pd.concat([results_df[keys], abn.rename('abn')], axis=1)
          .groupby(keys)['abn']
          .agg(valid_n=lambda x: x.notna().sum(),
               abn_any=lambda x: x.any(skipna=True))
          .reset_index()
    )

    # exclude treatments where this feature had no usable value
    grp = grp[grp['valid_n'] > 0]

    # counts
    abn_count  = int((grp['abn_any'] == True).sum())
    norm_count = int((grp['abn_any'] == False).sum())

    feature_counts.append({
        'feature':  z[:-2],   # strip '_z'
        'abnormal': abn_count,
        'normal':   norm_count
    })

# 4) Result table
feature_status_df = pd.DataFrame(feature_counts)

feature_status_df

,feature,abnormal,normal
0,ALP(IU/L),64,46
1,TC(mg/dL),66,44
2,TG(mg/dL),68,42
3,PL(mg/dL),68,42
4,TBIL(mg/dL),70,39
5,DBIL(mg/dL),38,32
6,GLC(mg/dL),62,48
7,BUN(mg/dL),71,39
8,CRE(mg/dL),16,45
9,Na(meq/L),47,61


In [92]:
# 1) Which features to score? (columns 11 onward)
feature_cols = real.columns[11:]

results_gen_z = []

# 2) Loop over each synthetic‐control block (compound, targetTime)
for (gen_cmpd, gen_time), gen_ctrl_group in gen.groupby(['COMPOUND_NAME','targetTime']):
    # need at least two control samples to compute an SD
    if gen_ctrl_group.shape[0] < 2:
        continue

    # 2a) compute control means & SDs for this compound/time
    ctrl_means = gen_ctrl_group[feature_cols].mean()
    ctrl_sds   = gen_ctrl_group[feature_cols].std(ddof=1)

    # 3) for each treatment dose (Low, Middle, High)
    for real_dose in ['High']:
        real_treat = real.loc[
            (real['COMPOUND_NAME']    == gen_cmpd) &
            (real['SACRIFICE_PERIOD'] == gen_time)   &
            (real['DOSE_LEVEL']       == real_dose)
        ]
        if real_treat.empty:
            continue

        # 4) compute z‑scores for each individual sample
        for _, sample in real_treat.iterrows():
            rec = {
                'compound':       gen_cmpd,
                'time':           gen_time,
                'dose':           real_dose,
                'INDIVIDUAL_ID':  sample['INDIVIDUAL_ID']
            }
            for feat in feature_cols:
                x     = sample[feat]
                mu    = ctrl_means[feat]
                sigma = ctrl_sds[feat]

                # guard against zero or missing sd
                if pd.isna(x) or pd.isna(mu) or pd.isna(sigma) or sigma == 0:
                    rec[f'{feat}_z'] = np.nan
                else:
                    rec[f'{feat}_z'] = (x - mu) / sigma

            results_gen_z.append(rec)

# 5) assemble into a DataFrame
results_gen_z_df = pd.DataFrame(results_gen_z)

In [93]:

best_thresh = threshold.set_index('feature')['best_threshold'].to_dict()

z_cols_gen = [c for c in results_gen_z_df.columns if c.endswith('_z')]
keys = ['compound','time','dose']

feature_counts_gen = []

for z in z_cols_gen:
    feat = z[:-2]
    thr  = best_thresh.get(feat, 2)

    # Skip if this feature isn't present on the REAL side
    if z not in results_df.columns:
        continue

    # --- GEN: cell-level abnormal (NA-aware)
    s_gen = results_gen_z_df[z]
    abn   = s_gen.abs().gt(thr)
    abn   = abn.where(s_gen.notna(), pd.NA).astype('boolean')

    grp_gen = (
        pd.concat([results_gen_z_df[keys], abn.rename('abn_gen')], axis=1)
          .groupby(keys)['abn_gen']
          .agg(valid_n_gen=lambda x: x.notna().sum(),
               abn_gen_any=lambda x: x.any(skipna=True))
          .reset_index()
    )

    # --- REAL: per-group valid counts for the same feature
    grp_real_valid = (
        results_df[keys + [z]]
        .groupby(keys, as_index=False)[z]
        .count()
        .rename(columns={z: 'valid_n_real'})
    )

    # --- Intersect validity: keep only groups valid on BOTH sides
    merged = grp_gen.merge(grp_real_valid, on=keys, how='left')
    merged['valid_n_real'] = merged['valid_n_real'].fillna(0).astype(int)
    eligible = merged[(merged['valid_n_real'] > 0) & (merged['valid_n_gen'] > 0)]

    # Counts among eligible groups
    abn_count  = int((eligible['abn_gen_any'] == True).sum())
    norm_count = int((eligible['abn_gen_any'] == False).sum())

    feature_counts_gen.append({
        'feature':   feat,
        'threshold': thr,
        'abnormal':  abn_count,
        'normal':    norm_count,
    })

feature_status_gen_df = pd.DataFrame(feature_counts_gen)

feature_status_gen_df

,feature,threshold,abnormal,normal
0,ALP(IU/L),6,50,60
1,TC(mg/dL),3,77,33
2,TG(mg/dL),3,77,33
3,PL(mg/dL),4,69,41
4,TBIL(mg/dL),2,70,39
5,DBIL(mg/dL),2,46,24
6,GLC(mg/dL),3,86,24
7,BUN(mg/dL),3,103,7
8,CRE(mg/dL),9,4,57
9,Na(meq/L),5,45,63


In [94]:
# keys shared by both frames
keys = ['compound','time','dose']

# ── 1) REAL flags (NA-aware; unchanged) ──
real_chunks = []
for z in [c for c in results_df.columns if c.endswith('_z')]:
    feat = z[:-2]
    s = results_df[z]

    abn = s.abs().gt(2)
    abn = abn.where(s.notna(), pd.NA).astype('boolean')

    grp = (
        pd.concat([results_df[keys], abn.rename('abn_real')], axis=1)
          .groupby(keys)['abn_real']
          .agg(valid_n_real=lambda x: x.notna().sum(),
               abn_real_any=lambda x: x.any(skipna=True))
          .reset_index()
    )
    # only keep groups where REAL had ≥1 usable value for this feature
    grp = grp[grp['valid_n_real'] > 0]
    grp['feature'] = feat
    real_chunks.append(grp)

real_flags_df = pd.concat(real_chunks, ignore_index=True)

# ── 2) GEN flags (NA-aware; aligned to REAL validity) ──
gen_chunks = []
for z in [c for c in results_gen_z_df.columns if c.endswith('_z')]:
    feat = z[:-2]
    thr  = best_thresh.get(feat, 2)

    s = results_gen_z_df[z]
    abn = s.abs().gt(thr)
    abn = abn.where(s.notna(), pd.NA).astype('boolean')

    grp = (
        pd.concat([results_gen_z_df[keys], abn.rename('abn_gen')], axis=1)
          .groupby(keys)['abn_gen']
          .agg(valid_n_gen=lambda x: x.notna().sum(),
               abn_gen_any=lambda x: x.any(skipna=True))
          .reset_index()
    )
    # DO NOT drop gen groups just because gen had all-NaN;
    # instead mark abn_gen_any as <NA> when valid_n_gen == 0
    grp.loc[grp['valid_n_gen'] == 0, 'abn_gen_any'] = pd.NA
    grp['abn_gen_any'] = grp['abn_gen_any'].astype('boolean')

    grp['feature']   = feat
    grp['threshold'] = thr
    gen_chunks.append(grp)

gen_flags_df = pd.concat(gen_chunks, ignore_index=True)

# ── 3) Merge, keeping rows where REAL had ≥1 valid sample.
#     (GEN may be <NA>; those rows won’t contribute to TP/TN/FP/FN unless you tally NA mismatches separately.)
merged_flags = (
    real_flags_df.merge(gen_flags_df, on=keys+['feature'], how='inner')
                 .query('valid_n_real > 0')  # <- ONLY real validity enforced
                 # (optional) if you want to also require gen validity, add: and valid_n_gen > 0
)

# ── 4) Confusion per feature (nullable-boolean safe) ──
def as_bool(series):
    return series == True, series == False  # NA becomes neither mask

# ── 4) Confusion per feature (treat gen NA as False) ──
confusion = []
for feat, grp in merged_flags.groupby('feature'):
    grp2 = grp.copy()
    # fill NA on generated side with False
    grp2['abn_gen_any_filled'] = grp2['abn_gen_any'].fillna(False)

    r = grp2['abn_real_any'].astype(bool)
    g = grp2['abn_gen_any_filled'].astype(bool)

    TP = int(( r &  g).sum())
    TN = int((~r & ~g).sum())
    FP = int((~r &  g).sum())
    FN = int(( r & ~g).sum())

    # optional: how many gen NAs were filled for this feature
    na_filled = int(grp2['abn_gen_any'].isna().sum())

    confusion.append({
        'feature': feat,
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'na_filled_gen': na_filled  # optional column
    })

confusion_df = pd.DataFrame(confusion)



In [95]:
# assume confusion_df has columns ['feature','TP','TN','FP','FN']

# 1) Compute total cases per feature
confusion_df['total'] = (
    confusion_df['TP'] +
    confusion_df['TN'] +
    confusion_df['FP'] +
    confusion_df['FN']
)

# 2) Compute accuracy = (TP + TN) / total
confusion_df['accuracy'] = (
    (confusion_df['TP'] + confusion_df['TN']) /
    confusion_df['total']
)

# 3) (Optional) drop the 'total' column if you don’t need it
confusion_df = confusion_df.drop(columns=['total'])


In [96]:
confusion_df

,feature,TP,TN,FP,FN,na_filled_gen,accuracy
0,A/G,41,25,17,13,0,0.687500
1,ALP(IU/L),40,36,10,24,0,0.690909
2,ALT(IU/L),70,12,27,1,0,0.745455
3,APTT(s),50,20,15,25,0,0.636364
4,AST(IU/L),42,30,17,21,0,0.654545
5,BUN(mg/dL),69,5,34,2,0,0.672727
6,Bas(%),0,16,0,0,0,1.000000
7,CRE(mg/dL),3,44,1,13,0,0.770492
8,Ca(mg/dL),43,27,14,26,0,0.636364
9,Cl(meq/L),39,15,44,11,0,0.495413


In [97]:
# --- Params ---
feat = "Cl(meq/L)"
thr  = best_thresh.get(feat, 2)

keys = ['compound','time','dose']
zcol_real = f'{feat}_z'
zcol_gen  = f'{feat}_z'

def max_abs_safe(s: pd.Series):
    s = s.dropna()
    return np.nan if s.empty else float(np.abs(s).max())

# --- Real: per-treatment z-aggregates + abnormal flag (|z|>2) ---
real_agg = (
    results_df[keys + [zcol_real]]
    .groupby(keys, as_index=False)
    .agg(
        z_real_list    = (zcol_real, lambda s: s.dropna().tolist()),
        max_abs_z_real = (zcol_real, max_abs_safe),
        mean_z_real    = (zcol_real, 'mean'),
        n_real         = (zcol_real, 'count')
    )
)
real_agg['abn_real'] = real_agg['max_abs_z_real'].abs() > 2

# --- Generated: per-treatment z-aggregates + abnormal flag (|z|>thr) ---
gen_agg = (
    results_gen_z_df[keys + [zcol_gen]]
    .groupby(keys, as_index=False)
    .agg(
        z_gen_list    = (zcol_gen, lambda s: s.dropna().tolist()),
        max_abs_z_gen = (zcol_gen, max_abs_safe),
        mean_z_gen    = (zcol_gen, 'mean'),
        n_gen         = (zcol_gen, 'count')
    )
)
gen_agg['abn_gen'] = gen_agg['max_abs_z_gen'].abs() > thr

# --- Merge real/gen z-aggregates ---
cmp = real_agg.merge(gen_agg, on=keys, how='inner')

# ==========================================================
# Add CONTROL mean/variance for THIS feature per (compound, time)
# (no GROUPS needed; attach to every (compound,time,dose) after merge)
#   - From real (DOSE_LEVEL == 'Control'): ctrl_mean_real, ctrl_var_real
#   - From gen  (synthetic controls at targetTime): ctrl_mean_gen,  ctrl_var_gen
# ==========================================================

# From REAL data: compute control stats per (compound,time)
# Requires a DataFrame `real` with columns: COMPOUND_NAME, SACRIFICE_PERIOD, DOSE_LEVEL, and feature columns
_real_ctrl = (
    real.loc[real['DOSE_LEVEL'] == 'Control', ['COMPOUND_NAME', 'SACRIFICE_PERIOD', feat]]
        .groupby(['COMPOUND_NAME', 'SACRIFICE_PERIOD'])
        .agg(
            ctrl_mean_real = (feat, 'mean'),
            ctrl_var_real  = (feat, lambda s: s.std(ddof=1) if s.size > 1 else np.nan)
        )
        .reset_index()
        .rename(columns={'COMPOUND_NAME': 'compound', 'SACRIFICE_PERIOD': 'time'})
)

# From GENERATED data: compute synthetic-control stats per (compound,time)
# Requires a DataFrame `gen` with columns: COMPOUND_NAME, targetTime, and feature columns
_gen_ctrl = (
    gen[['COMPOUND_NAME', 'targetTime', feat]]
        .groupby(['COMPOUND_NAME', 'targetTime'])
        .agg(
            ctrl_mean_gen = (feat, 'mean'),
            ctrl_var_gen  = (feat, lambda s: s.std(ddof=1) if s.size > 1 else np.nan)
        )
        .reset_index()
        .rename(columns={'COMPOUND_NAME': 'compound', 'targetTime': 'time'})
)

# Merge control stats onto cmp (replicates across each dose at that (compound,time))
cmp = (
    cmp.merge(_real_ctrl, on=['compound','time'], how='left')
       .merge(_gen_ctrl,  on=['compound','time'], how='left')
)

# --- Find treatment-level mismatches ---
mismatches = cmp.loc[cmp['abn_real'] != cmp['abn_gen']].copy()

mismatches['threshold_gen'] = thr
mismatches['error_type'] = np.select(
    [
        (~mismatches['abn_real']) & (mismatches['abn_gen']),  # FP
        (mismatches['abn_real'])  & (~mismatches['abn_gen'])  # FN
    ],
    ['FP', 'FN'],
    default='(other)'
)

# Include the new control mean/var columns
mismatches = mismatches[
    keys + [
        'abn_real','abn_gen','error_type','threshold_gen',
        'n_real','n_gen',
        'max_abs_z_real','max_abs_z_gen',
        'mean_z_real','mean_z_gen',
        'z_real_list','z_gen_list',
        'ctrl_mean_real','ctrl_var_real',
        'ctrl_mean_gen','ctrl_var_gen'
    ]
].sort_values(keys)

mismatches  # <- disagreements + control mean/var for the given feature, per (compound,time,dose)


,compound,time,dose,abn_real,abn_gen,error_type,threshold_gen,n_real,n_gen,max_abs_z_real,max_abs_z_gen,mean_z_real,mean_z_gen,z_real_list,z_gen_list,ctrl_mean_real,ctrl_var_real,ctrl_mean_gen,ctrl_var_gen
0,amiodarone,15 day,High,False,True,FP,3,5,5,0.923133,4.124257,-2.637522e-01,-3.298746e+00,"[-0.9231326627541057, 0.39562828403746847, 0.3...","[-4.124257498216263, -2.4732336798750834, -2.4...",101.40,1.516575,104.996000,1.211370
1,amiodarone,29 day,High,False,True,FP,3,4,4,1.258396,5.433867,-4.256341e-01,-3.329529e+00,"[0.22206996305927948, -1.258396457335931, -0.1...","[-1.6928224532479499, -5.4338665488235245, -2....",102.40,2.701851,104.810000,1.069220
2,amiodarone,4 day,High,False,True,FP,3,5,5,1.129368,4.060075,4.343722e-01,-6.641014e-01,"[-0.6081211398682996, 0.2606233456578392, 0.26...","[-4.060074621099688, -1.2300969576937963, -1.2...",103.40,2.302173,104.869333,0.706719
3,amiodarone,8 day,High,False,True,FP,3,5,5,0.526235,4.037039,-8.770580e-02,-2.896633e+00,"[-0.08770580193070417, -0.08770580193070417, 0...","[-2.896632794569062, -2.896632794569062, -1.75...",103.20,2.280351,105.540000,0.876880
4,amitriptyline,15 day,High,False,True,FP,3,5,5,1.224745,4.239718,-8.164966e-01,-3.357179e+00,"[-0.8164965809277261, -1.2247448713915892, -0....","[-3.3571786436227615, -4.239717719233368, -3.3...",106.00,2.449490,107.804000,1.133094
5,amitriptyline,29 day,High,False,True,FP,3,3,3,1.643168,4.632758,-1.034587e+00,-3.763698e+00,"[-1.6431676725154958, -1.6431676725154958, 0.1...","[-4.632757649490626, -4.632757649490626, -2.02...",104.80,1.095445,106.553846,0.767112
12,benzbromarone,15 day,High,False,True,FP,3,5,5,1.592607,4.417492,-3.781009e-01,2.679191e+00,"[-1.5926068113410878, -0.3895585006158013, -0....","[0.4856205659345968, 2.6584966777158687, 2.761...",108.08,1.745566,104.830667,0.966461
13,benzbromarone,29 day,High,False,True,FP,3,4,4,1.917933,5.427240,1.163907e+00,4.071919e+00,"[0.2938769068226259, 1.917933497158216, 1.3765...","[2.508087878138171, 5.427240040341782, 4.45418...",106.72,1.293058,105.295714,0.719387
15,benzbromarone,8 day,High,False,True,FP,3,4,4,0.639356,4.866416,2.187269e-01,4.061860e+00,"[0.6393556558250804, -0.4542790186125678, 0.05...","[4.866416019538509, 2.774569738111953, 3.74003...",106.54,1.188697,104.275714,0.621461
17,benziodarone,29 day,High,False,True,FP,3,5,5,1.582513,3.710391,5.275044e-01,1.797818e+00,"[0.26375218935831857, 0.9231326627541057, 1.58...","[1.3196750599918146, 2.5150329041872994, 3.710...",105.60,1.516575,104.896000,0.836570


In [98]:

# --- Setup
keys = ['compound','time','dose']
dbil_feat = "Cl(meq/L)"
zcol = f"{dbil_feat}_z"

# (Optional but recommended) normalize key dtypes/whitespace
for df in (results_df, results_gen_z_df):
    for k in keys:
        df[k] = df[k].astype(str).str.strip()

# --- Per-group valid counts (non-NaN) for DBIL_z
real_valid = (
    results_df[keys + [zcol]]
      .groupby(keys, as_index=False)[zcol].count()
      .rename(columns={zcol: 'valid_n_real'})
)

gen_valid = (
    results_gen_z_df[keys + [zcol]]
      .groupby(keys, as_index=False)[zcol].count()
      .rename(columns={zcol: 'valid_n_gen'})
)

# --- Also capture total generated rows per group (regardless of NaN)
gen_total = (
    results_gen_z_df[keys]
      .groupby(keys, as_index=False)
      .size()
      .rename(columns={'size':'gen_group_size'})
)

# --- Merge side-by-side
cmp = (
    real_valid.merge(gen_valid, on=keys, how='left')
              .merge(gen_total, on=keys, how='left')
)

# Fill missing gen columns with 0 -> means group absent in gen
cmp['valid_n_gen']   = cmp['valid_n_gen'].fillna(0).astype(int)
cmp['gen_group_size'] = cmp['gen_group_size'].fillna(0).astype(int)
cmp['valid_n_real']   = cmp['valid_n_real'].astype(int)

# --- Diagnose discrepancies where REAL had data but GEN did not
dbil_diff = cmp.query('valid_n_real > 0 and valid_n_gen == 0').copy()
dbil_diff['reason'] = np.where(
    dbil_diff['gen_group_size'] == 0,
    'missing_in_generated',      # no rows for this (compound,time,dose) in gen
    'all_NaN_in_generated'       # rows exist, but DBIL_z all NaN
)

# --- Quick tallies
n_real_groups = (cmp['valid_n_real'] > 0).sum()
n_gen_groups  = ((cmp['valid_n_real'] > 0) & (cmp['valid_n_gen'] > 0)).sum()
summary = {
    'real_groups_with_DBIL': int(n_real_groups),
    'gen_groups_eligible'  : int(n_gen_groups),
    'hard_na_or_missing'   : int(len(dbil_diff)),
    'by_reason'            : dbil_diff['reason'].value_counts().to_dict()
}

print(summary)
print("Examples of discrepant groups (up to 10):")
print(dbil_diff.head(10).to_string(index=False))


{'real_groups_with_DBIL': 109, 'gen_groups_eligible': 109, 'hard_na_or_missing': 0, 'by_reason': {}}
Examples of discrepant groups (up to 10):
Empty DataFrame
Columns: [compound, time, dose, valid_n_real, valid_n_gen, gen_group_size, reason]
Index: []


In [99]:
feat = "DBIL(mg/dL)"
zcol = f"{feat}_z"
sus = results_df.query(
    "(compound == 'carboplatin' and time == '29 day' and dose == 'High') or "
    "(compound == 'iproniazid' and time == '29 day' and dose == 'High')"
)
cols_to_show = [c for c in [ 'compound','time','dose', feat, zcol, 'vehicle','Vehicle','lab','Lab'] if c in sus.columns]
print(sus[cols_to_show])


        compound    time  dose  DBIL(mg/dL)_z
118  carboplatin  29 day  High       6.260990
119  carboplatin  29 day  High      -0.447214
120  carboplatin  29 day  High      -0.447214
121  carboplatin  29 day  High      10.733126
122  carboplatin  29 day  High      -0.447214
310   iproniazid  29 day  High      -0.447214
311   iproniazid  29 day  High       4.024922
312   iproniazid  29 day  High      -0.447214
313   iproniazid  29 day  High      -0.447214
314   iproniazid  29 day  High      -0.447214


In [100]:
#confusion_df.to_csv('/account001/mansi.chandra/clin_path/results_plots_updated/contrastive_high_threshold.csv', index=False)

In [101]:
gen['COMPOUND_NAME'].unique()

array(['amiodarone', 'amitriptyline', 'amphotericin B', 'benzbromarone',
       'benziodarone', 'bromobenzene', 'carboplatin', 'cephalothin',
       'chlormadinone', 'chlorpromazine', 'cisplatin', 'cyclosporine A',
       'diltiazem', 'enalapril', 'gemfibrozil', 'imipramine',
       'iproniazid', 'ketoconazole', 'lomustine', 'lornoxicam',
       'mefenamic acid', 'meloxicam', 'phenobarbital', 'promethazine',
       'rosiglitazone maleate', 'thioacetamide', 'thioridazine',
       'triazolam'], dtype=object)

In [102]:
real[(real['COMPOUND_NAME'] == 'cyclosporine A')&(real['DOSE_LEVEL'] == 'Control') & (real['SACRIFICE_PERIOD'] == '4 day')].iloc[:, 11:]

,ALP(IU/L),TC(mg/dL),TG(mg/dL),PL(mg/dL),TBIL(mg/dL),DBIL(mg/dL),GLC(mg/dL),BUN(mg/dL),CRE(mg/dL),Na(meq/L),...,Plat(x10_4/uL),WBC(x10_2/uL),Neu(%),Eos(%),Bas(%),Mono(%),Lym(%),PT(s),APTT(s),Fbg(mg/dL)
477,1509,64,59.0,121,0.01,0.0,187,14.0,0.1,140.0,...,131.4,90.9,12.0,0.0,0.0,2.0,85.0,16.9,21.6,268.0
478,1809,85,75.0,149,0.00,0.0,204,15.0,0.2,142.0,...,112.8,92.4,17.0,1.0,0.0,3.0,79.0,16.4,21.7,309.0
479,2005,91,207.0,180,0.00,0.0,227,20.0,0.1,139.0,...,121.8,92.8,12.0,1.0,0.0,2.0,85.0,16.6,20.8,332.0
480,1646,64,69.0,117,0.00,0.0,207,15.0,0.2,142.0,...,113.9,85.9,8.0,1.0,0.0,2.0,88.0,17.2,22.8,297.0
481,2055,86,116.0,157,0.00,0.0,185,17.0,0.2,143.0,...,133.0,78.8,10.0,1.0,0.0,3.0,85.0,15.7,23.4,302.0


In [103]:
gen[(gen['COMPOUND_NAME'] == 'cyclosporine A')&(gen['DOSE_LEVEL'] == 'High') & (gen['targetTime'] == '4 day')].iloc[:, 18:]

,ALP(IU/L),TC(mg/dL),TG(mg/dL),PL(mg/dL),TBIL(mg/dL),DBIL(mg/dL),GLC(mg/dL),BUN(mg/dL),CRE(mg/dL),Na(meq/L),...,Plat(x10_4/uL),WBC(x10_2/uL),Neu(%),Eos(%),Bas(%),Mono(%),Lym(%),PT(s),APTT(s),Fbg(mg/dL)
3425,1544,77.381510,89.677734,141.33757,0.01,0.00,193.99867,17,0.2,141.6,...,128.708680,90.336630,15.049395,0.884547,0.131549,2.312721,80.505950,16.099705,21.935854,296.67440
3426,1469,71.159810,83.803950,139.26476,0.01,0.01,195.48193,16,0.2,142.2,...,124.149080,89.635990,14.555665,0.886614,0.188428,2.274700,81.363590,16.414629,22.473250,298.58750
3427,1506,78.384090,88.470245,144.52496,0.01,0.02,193.33203,16,0.2,142.0,...,129.027440,90.410540,16.774221,0.978344,0.054607,2.457310,78.502144,16.229320,22.277815,297.62350
3428,1487,75.791450,70.535675,138.87862,0.01,0.00,192.34258,17,0.2,142.8,...,123.664520,91.843544,13.857878,0.932198,0.019681,2.548171,81.883575,16.158663,22.632570,298.58150
3429,1532,72.749760,88.026825,141.40538,0.01,0.01,195.75485,16,0.2,142.4,...,134.471620,107.207130,15.816009,0.918457,0.098989,2.634734,79.650080,16.178950,22.116697,302.39523
3430,1542,76.102490,83.600200,141.31421,0.01,0.01,194.03072,16,0.2,141.7,...,129.776290,82.700010,15.372733,0.865611,0.188301,2.262291,80.236300,16.184443,22.021038,297.63828
3431,1448,72.751170,79.466430,138.61674,0.01,0.00,195.81921,16,0.2,142.2,...,129.586670,99.013824,14.454117,0.877895,0.014480,2.306740,81.379920,16.450110,22.378790,298.54388
3432,1523,82.686790,83.828210,145.60368,0.01,0.01,192.06044,16,0.2,142.1,...,120.656380,82.939964,16.568802,0.970313,0.250399,2.594272,78.528310,16.238420,21.841380,298.66537
3433,1457,75.720505,69.648415,139.93205,0.01,0.02,191.98914,17,0.3,142.7,...,116.661240,85.317410,13.690993,0.935566,0.035346,2.683551,81.870160,16.167320,22.335000,298.43723
3434,1516,68.868240,82.323300,134.56119,0.01,0.02,196.27864,16,0.2,142.5,...,126.180500,105.810380,15.839043,0.932768,0.104482,2.601346,79.707600,16.246075,22.102959,301.18567
